# Instrucciones:

1.	Modifica el codigo brindado para integrar la métrica de perplejidad y de coherencia.
Investiga las funciones:
- log_perplexity
- CoherenceModel – getCoherence

2.	Prueba agregando los parámetros correspondientes a alfa y beta al crear el modelo LDA. Prueba con diferentes valores para cada parámetro. También utiliza diferentes valores de num_topics.

3.	Evalúa cada combinación de estos 3 parametros con las métricas implementadas.

4.	Describe el funcionamiento de pyLDAVis, así como la interpretación de lo que se visualiza.

5.	Elabora tu reporte con los hallazgos. Sube tu reporte en PDF. Comparte las modificaciones a tu código.


# Instalación de bibliotecas

In [ ]:
!pip install gensim
!pip install pyLDAvis
!python -m spacy download es_core_news_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 38.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 117.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


# Importación de bibliotecas

In [ ]:
import pyLDAvis.gensim_models as gensimvis
import pyLDAvis
import gensim
from gensim import corpora
from gensim.models import LdaModel
from gensim.models import CoherenceModel
import numpy as np
import pandas as pd
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import spacy
nlp = spacy.load("es_core_news_sm")
import nltk
nltk.download('punkt')
nltk.download('stopwords')
import warnings

warnings.filterwarnings("ignore",category=DeprecationWarning,module="jupyter_client")
def separador():
    print("\n" + "=" * 170 + "\n")
def header(Titulo):
  por=((170-len(Titulo))//10)
  a=int(1.2*por)
  print("."*a+"·"*a+"~"*a+"≈"*a+"≋"
  *int(por*.2),Titulo,"≋"*int(por*.2)
  +"≈"*a+"~"*a+"·"*a+"."*a,"\n\n")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# Texto proporcionado

In [ ]:

documents = [
    # Política
    "El presidente anunció nuevas reformas durante la conferencia en el Congreso.",
    "Los partidos políticos debatieron sobre el futuro del sistema electoral.",
    "La oposición critica la gestión gubernamental en materia de seguridad.",
    "Miles de personas protestaron en la capital contra la nueva ley propuesta.",
    "Se firmó un acuerdo internacional para fomentar la cooperación política.",

    # Economía
    "El banco central decidió aumentar las tasas de interés para controlar la inflación.",
    "La economía del país creció un 3% durante el segundo trimestre del año.",
    "Las exportaciones de productos agrícolas aumentaron notablemente.",
    "El desempleo bajó por tercer mes consecutivo, según datos oficiales.",
    "Los inversores mostraron confianza ante las nuevas políticas económicas.",

    # Deportes
    "La selección nacional clasificó a la final del campeonato continental.",
    "El delantero estrella fue transferido al club europeo por una suma millonaria.",
    "Los fanáticos celebraron la victoria con una caravana en las calles.",
    "El equipo local perdió el partido decisivo por un gol en el último minuto.",
    "La liga anunció cambios en el reglamento para la próxima temporada.",

    # Tecnología
    "La empresa lanzó un nuevo smartphone con inteligencia artificial integrada.",
    "Se descubrió una vulnerabilidad crítica en el sistema operativo.",
    "La inversión en energías renovables incluye avances en baterías inteligentes.",
    "El uso de la automatización ha transformado la industria manufacturera.",
    "Expertos debaten sobre los riesgos éticos del desarrollo de la IA.",

    # Medio ambiente
    "El cambio climático está afectando los patrones de lluvia en la región.",
    "Un nuevo informe advierte sobre la pérdida acelerada de biodiversidad.",
    "Se implementaron miles de políticas públicas para reducir las emisiones de carbono.",
    "Miles de voluntarios participaron en una jornada de reforestación.",
    "Organizaciones internacionales exigen medidas urgentes contra la contaminación."]


# Preprocesamiento del texto y modelado con un grid de valores

In [ ]:
#se lematiza, porque en el video se recomienda en el preprocesamiento del texto
texts = []

for doc in documents:
    doc_spacy = nlp(doc)
    lemmatized_tokens = []
    stop_words=set(stopwords.words("spanish"))
    for token in doc_spacy:
      if token.lemma_ not in stop_words and token.lemma_.isalpha():
        lemmatized_tokens.append(token.lemma_.lower())

    texts.append(lemmatized_tokens)

dictionary=corpora.Dictionary(texts)
corpus=[dictionary.doc2bow(text) for text in texts]

#==============================================================

#Valores que se van a probar en el grid
lista_topicos = [4, 5, 6]
lista_a = [0.1,.3, 0.5,.8]
lista_b = [0.1,.3, 0.5,.8]
header('Combinación de valores')
#mejores resultados
resultados = []
conteo=(1)#para saber cuantas combinaciones se buscan
#Grid para buscar los mejores valores
for k in lista_topicos:
    for a in lista_a:
        for b in lista_b:

            lda_model = LdaModel(corpus=corpus,
                                 id2word=dictionary,
                                 num_topics=k,
                                 random_state=9,
                                 alpha=a,
                                 eta=b,
                                 passes=15)

            log_perplexity = lda_model.log_perplexity(corpus) #PERPLEJIDAD
            perplexity = np.exp(-1 * log_perplexity)#siendo log, el valor menor es el mejor
            #COHERENCIA
            coherence_model = CoherenceModel(model=lda_model,
                                             texts=texts,
                                             dictionary=dictionary,
                                             coherence='c_v')

            coherence = coherence_model.get_coherence()
            #se van guardando los resultados de cada combinación
            resultados.append({
                'Topics': k,
                'Alpha': a,
                'Beta (eta)': b,
                'LogPerplexity': log_perplexity,
                'Perplexity': perplexity,
                'Coherence (C_v)': coherence})

            print(f"Prueba {conteo} - Parámetros: k={k}, a={a}, b={b} con una Coherencia: {coherence:.4f} y Perplejidad: {perplexity:.4f}")
            conteo+=1

resultados_df = pd.DataFrame(resultados)
header('Resultados')
print(resultados_df.sort_values(by='Coherence (C_v)', ascending=False))#se imprime el df ordenado por coherencia (descendiente)

best_model_config = resultados_df.loc[resultados_df['Coherence (C_v)'].idxmax()] #el mejor valor es el primero
header('Mejor Configuración')
print(f"Mejor Configuración Encontrada (basada en Coherencia\n{best_model_config}")
separador()
mejor_k = int(best_model_config['Topics'])
mejor_a = best_model_config['Alpha']
mejor_b = best_model_config['Beta (eta)']

#se entrena el modelo de nuevo, pero ahora con los mejores parametros
best_lda_model = LdaModel(corpus=corpus,
                          id2word=dictionary,
                          num_topics=mejor_k,
                          random_state=9,
                          alpha=mejor_a,
                          eta=mejor_b,
                          passes=15)


header('Topicos del mejor modelo')
for idx, topic in best_lda_model.print_topics():
  print(f"Tópico {idx}: {topic}\n")
separador()


................················~~~~~~~~~~~~~~~~≈≈≈≈≈≈≈≈≈≈≈≈≈≈≈≈≋≋ Combinación de valores ≋≋≈≈≈≈≈≈≈≈≈≈≈≈≈≈≈≈~~~~~~~~~~~~~~~~················................ 


Prueba 1 - Parámetros: k=4, a=0.1, b=0.1 con una Coherencia: 0.4257 y Perplejidad: 398.0763
Prueba 2 - Parámetros: k=4, a=0.1, b=0.3 con una Coherencia: 0.3667 y Perplejidad: 266.4654
Prueba 3 - Parámetros: k=4, a=0.1, b=0.5 con una Coherencia: 0.3747 y Perplejidad: 238.6991
Prueba 4 - Parámetros: k=4, a=0.1, b=0.8 con una Coherencia: 0.3648 y Perplejidad: 222.9183
Prueba 5 - Parámetros: k=4, a=0.3, b=0.1 con una Coherencia: 0.4068 y Perplejidad: 430.8668
Prueba 6 - Parámetros: k=4, a=0.3, b=0.3 con una Coherencia: 0.3557 y Perplejidad: 294.2915
Prueba 7 - Parámetros: k=4, a=0.3, b=0.5 con una Coherencia: 0.3246 y Perplejidad: 265.5233
Prueba 8 - Parámetros: k=4, a=0.3, b=0.8 con una Coherencia: 0.3219 y Perplejidad: 249.1270
Prueba 9 - Parámetros: k=4, a=0.5, b=0.1 con una Coherencia: 0.3781 y Perplejidad: 460.8848
Prueba 10 - 

#Se exporta a html

In [ ]:
vis = gensimvis.prepare(best_lda_model, corpus, dictionary)
pyLDAvis.save_html(vis, "/content/drive/MyDrive/MAESTRIA/PLN/lda_optimo_output.html")

#Hallazgos

## Mejor modelo (Se encontraron 5 clusters)

    Mejor Configuración Encontrada (basada en Coherencia)
    Topics               5.000000
    Alpha                0.500000
    Beta (eta)           0.100000
    LogPerplexity       -6.236115
    Perplexity         510.870025
    Coherence (C_v)      0.436184
    Name: 24, dtype: float64



    Tópico 0: 0.071*"partido" + 0.071*"sistema" + 0.039*"político" + 0.037*"electoral" + 0.037*"futuro" + 0.037*"debatir" + 0.037*"último" + 0.037*"minuto" + 0.037*"perder" + 0.037*"gol"

    Tópico 1: 0.041*"cambio" + 0.021*"anunciar" + 0.021*"internacional" + 0.021*"patrón" + 0.021*"industria" + 0.021*"criticar" + 0.021*"región" + 0.021*"seguridad" + 0.021*"afectar" + 0.021*"oposición"

    Tópico 2: 0.027*"aumentar" + 0.026*"país" + 0.026*"crecer" + 0.026*"año" + 0.026*"economía" + 0.026*"trimestre" + 0.026*"notablemente" + 0.026*"agrícola" + 0.026*"segundo" + 0.026*"nacional"

    Tópico 3: 0.092*"nuevo" + 0.040*"mil" + 0.020*"política" + 0.020*"mostrar" + 0.020*"inversor" + 0.020*"confianza" + 0.020*"económico" + 0.020*"empresa" + 0.020*"artificial" + 0.020*"integrado"

    Tópico 4: 0.021*"ético" + 0.021*"ia" + 0.021*"riesgo" + 0.021*"experto" + 0.021*"debatar" + 0.021*"fomentar" + 0.021*"cooperación" + 0.021*"desarrollo" + 0.021*"acuerdo" + 0.021*"dato"



## Funciones log_perplexity y coherence_model
**- log_perplexity (perplejidad)**. Es una metrica estandar que mide que tan bien predice una muestra un modelo de probabilidad. En el caso de log_perplexity, es la log-verosimilidud.

Perplejidad=exp(−1×log_verosimilitud_por_palabra)

**- coherence_model**. Es una metrica diseñada especificamente para abordar la limitación de la perplejidad. Mide si las palabras que componen un tópico tienen sentido juntas desde una perspectiva humana.
Evalúa la interpretabilidad semántica de un tópico. Un tópico coherente es aquel cuyas palabras clave estan semáticamente relacionadas. Su interpretación es directa, un valor mas alto es mejor.


## PyLDAVis
PyLDAVis es una herramienta interactiva, que ayuda a visualizar los tópicos que se crearon cuando se aplica el modelo LDA. Comunmente, las listas de palabras de los tópicos no son suficientes, es ahí donde esta herramienta facilita la interpretabilidad.

La visualización se divide en dos grandes paneles:
1. Panel izquierdo. Muestra una visión en donde cada circulo es un tópico del tamaño que corresponde a su prevalencia (tema que aparece mas frecuentemente en los datos). La distancia representa la similitud entre los tópicos y cuando se sobreponen es porque coinciden en su vocabulario.

2. Panel derecho. muestra en azul la frecuencia total de una palabra en todos los documentos. La barra roja la frecuencia especificicamente en el documento seleccionado. La palabra es clave de un tema cuando aparece la barra roja larga y azul corta.

**Lambda**
En la parte de arriba a la derecha, hay un deslizador llamado lambda.
- λ=1: Muestra las palabras mas frecuentes del tópico.
- λ=0: Muestra las palabras mas distintivas o exclusivas de ese tópico.